In [2]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.linalg import block_diag
from astropy import units as u
from astropy.time import Time
from poliastro.bodies import Earth
from poliastro.twobody import Orbit, sampling
from poliastro.util import time_range

In [34]:
# Constants
mu_e    = 398600.44    # km**3/2**2
R_e     = 6378.137      # km
J2      = 1.08264e-3

PARAM_DIM = 10
STATE_DIM = 6
MEASUREMENT_DIM = 2
K = 181

nu = 1
i_max = 10
t_eval = np.arange(0, 1810, 10)
epsilon = 1e-5

# Turn generated points into elevation and azimuth
def transform_state(x_target, x_observer):
    r_obs = x_observer[:3]
    v_obs = x_observer[3:6]
    rel = x_target[:3] - r_obs

    R_hat = r_obs / np.linalg.norm(r_obs)
    H = np.cross(r_obs, v_obs)
    C_hat = H / np.linalg.norm(H)
    T_hat = np.cross(C_hat, R_hat)

    R_orbital = np.vstack((T_hat, C_hat, R_hat))
    rel_local = R_orbital @ rel

    x, y, z = rel_local
    azimuth = np.arctan2(y, x)
    elevation = np.arcsin(z / np.linalg.norm(rel_local))

    return np.array([azimuth, elevation])

def dynamics(t, state):
    r = state[:3]
    v = state[3:]

    # J2 Perturbation
    a_j2r = [mu_e * r[0] * J2 * R_e**2/np.linalg.norm(r)**5 * (-3/2 + 15/2 * r[2]**2/np.linalg.norm(r)**2),
            mu_e * r[1] * J2 * R_e**2/np.linalg.norm(r)**5 * (-3/2 + 15/2 * r[2]**2/np.linalg.norm(r)**2),
            mu_e * r[2] * J2 * R_e**2/np.linalg.norm(r)**5 * (-9/2 + 15/2 * r[2]**2/np.linalg.norm(r)**2)
            ]
    
    norm_r = np.linalg.norm(r)
    a = -mu_e/norm_r**3 * r + a_j2r 
    return np.concatenate([v, a])

def compute_stt(t_eval, X, propagate_with_impulse, epsilon_vector = None):
    if epsilon_vector is None:
        epsilon_vector = np.array([10, 10, 10, 1e-3, 1e-3, 1e-3, 5e-3, 5e-3, 5e-3, 5])  # r/v, Δv, t_impulse

    phi = np.zeros((STATE_DIM, PARAM_DIM))
    psi = np.zeros((STATE_DIM, PARAM_DIM, PARAM_DIM))
    phi_tmp = np.zeros((STATE_DIM, STATE_DIM))
    psi_tmp = np.zeros((STATE_DIM, STATE_DIM, STATE_DIM))
    A_1 = np.zeros((STATE_DIM, STATE_DIM))
    A_2 = np.zeros((STATE_DIM, STATE_DIM, STATE_DIM))
    I = np.identity(6)
    
    t_pre = t_eval[t_eval <= X[9]]
    t_post = t_eval[t_eval > X[9]]

    # Phi from t_0 - t (ie A_1)
    for j in range(STATE_DIM):
        epsilon_current = np.zeros((PARAM_DIM))
        epsilon_current[j] = epsilon_vector[j]
    
        x_plus_pre = propagate_with_impulse(t_pre, X + epsilon_current)[-1]
        x_minus_pre = propagate_with_impulse(t_pre, X - epsilon_current)[-1]
        A_1[:, j] = (x_plus_pre - x_minus_pre) / (2 * epsilon_vector[j])

    # Psi from t_0 - t (ie A_2)
    for j in range(STATE_DIM):
        for k in range(STATE_DIM):
            X_pp = X.copy()
            X_pp[j] += epsilon_vector[j]
            X_pp[k] += epsilon_vector[k]

            X_pm = X.copy()
            X_pm[j] += epsilon_vector[j]
            X_pm[k] -= epsilon_vector[k]

            X_mp = X.copy()
            X_mp[j] -= epsilon_vector[j]
            X_mp[k] += epsilon_vector[k]

            X_mm = X.copy()
            X_mm[j] -= epsilon_vector[j]
            X_mm[k] -= epsilon_vector[k]

            x_pp = propagate_with_impulse(t_pre, X_pp)[-1]
            x_pm = propagate_with_impulse(t_pre, X_pm)[-1]
            x_mp = propagate_with_impulse(t_pre, X_mp)[-1]
            x_mm = propagate_with_impulse(t_pre, X_mm)[-1]

            A_2[:, j, k] = (x_pp - x_pm - x_mp + x_mm) / (4 * epsilon_vector[j] * epsilon_vector[k])

    # t - t_2
    X_no_dv = np.hstack((X[:6], 0, 0, 0, 1800))
    X_t1 = np.hstack((propagate_with_impulse(t_pre, X)[-1], X[6:]))
    X_no_man = np.hstack((propagate_with_impulse(t_pre, X_no_dv)[-1], X[6:]))

    # f
    f_plus = propagate_with_impulse(t_post, X_no_man)[-1]
    f_minus = propagate_with_impulse(t_post, X_t1)[-1]

    # g
    g_minus_p = propagate_with_impulse(np.hstack(([t_post[0] + epsilon_vector[9]], t_post[1:])), X_no_man)[-1]
    g_minus_m = propagate_with_impulse(np.hstack(([t_post[0] - epsilon_vector[9]], t_post[1:])), X_no_man)[-1]
    g_minus = (g_minus_p - g_minus_m) / (2 * epsilon_vector[9])

    g_plus_p = propagate_with_impulse(np.hstack(([t_post[0] + epsilon_vector[9]], t_post[1:])), X_t1)[-1]
    g_plus_m = propagate_with_impulse(np.hstack(([t_post[0] - epsilon_vector[9]], t_post[1:])), X_t1)[-1]
    g_plus = (g_plus_p - g_plus_m) / (2 * epsilon_vector[9])

    # H
    H_plus = np.zeros((STATE_DIM, STATE_DIM))
    H_minus = np.zeros((STATE_DIM, STATE_DIM))

    for j in range(STATE_DIM):
        x_plus_p = propagate_with_impulse(t_pre, X_t1 + epsilon_vector)[-1]
        x_minus_p = propagate_with_impulse(t_pre, X_t1 - epsilon_vector)[-1]
        H_plus[:, j] = (x_plus_p - x_minus_p) / (2 * epsilon_vector[:6])

        x_plus_m = propagate_with_impulse(t_pre, X_no_man + epsilon_vector)[-1]
        x_minus_m = propagate_with_impulse(t_pre, X_no_man - epsilon_vector)[-1]
        H_minus[:, j] = (x_plus_m - x_minus_m) / (2 * epsilon_vector[:6])
    
    B_1 =  f_minus - f_plus
    B_2 = g_minus - g_plus + 2 * np.einsum('kp,p ->k', H_plus, f_plus)
    C = H_minus - np.einsum('kl,lp->kp', H_plus, A_1)
    D = -H_plus

    # Phi and Psi for t1 - t2
    for j in range(STATE_DIM):
        epsilon_current = np.zeros((PARAM_DIM))
        epsilon_current[j] = epsilon_vector[j]

        x_plus_post = propagate_with_impulse(t_post, X_t1 + epsilon_current)[-1]
        x_minus_post = propagate_with_impulse(t_post, X_t1 - epsilon_current)[-1]
        phi_tmp[:, j] = (x_plus_post - x_minus_post) / (2 * epsilon_vector[j])
    
    for j in range(STATE_DIM):
        for k in range(STATE_DIM):
            X_t1pp = X.copy()
            X_t1pp[j] += epsilon_vector[j]
            X_t1pp[k] += epsilon_vector[k]

            X_t1pm = X.copy()
            X_t1pm[j] += epsilon_vector[j]
            X_t1pm[k] -= epsilon_vector[k]

            X_t1mp = X.copy()
            X_t1mp[j] -= epsilon_vector[j]
            X_t1mp[k] += epsilon_vector[k]

            X_t1mm = X.copy()
            X_t1mm[j] -= epsilon_vector[j]
            X_t1mm[k] -= epsilon_vector[k]

            x_pp = propagate_with_impulse(t_post, X_t1pp)[-1]
            x_pm = propagate_with_impulse(t_post, X_t1pm)[-1]
            x_mp = propagate_with_impulse(t_post, X_t1mp)[-1]
            x_mm = propagate_with_impulse(t_post, X_t1mm)[-1]

            psi_tmp[:, j, k] = (x_pp - x_pm - x_mp + x_mm) / (4 * epsilon_vector[j] * epsilon_vector[k])

    # Compile matrices
    # Phi    for i in range(STATE_DIM):
    for i in range(STATE_DIM):
        phi[:, i] = phi_tmp @ A_1[:, i]
    phi[:, 6:9] = phi_tmp[:, 3:6]
    phi[:, 9] = phi_tmp @ B_1

    # Psi
    psi[:, 6:9, 6:9] = psi_tmp[:, 3:6, 3:6]

    # 0 - 6 x 0 - 6
    for i in range(STATE_DIM):
        for j in range(STATE_DIM):
            psi[:, i, j] = np.einsum('ijk, j, k->i', psi_tmp, A_1[:, i], A_1[:, j]) + phi_tmp @ A_2[:, i, j]

        for j in range(6, 9):
            psi[:, i, j - 3] = np.einsum('ijk, j, k->i', psi_tmp, A_1[:, i], I[:, j - 3])
            psi[:, j, i - 3] = np.einsum('ijk, j, k->i', psi_tmp, A_1[:, i], I[:, j - 3])

        psi[:, i, 9] = np.einsum('ijk, j, k->i', psi_tmp, A_1[:, i], B_1) + phi_tmp @ C[:, i]
        psi[:, 9, i] = np.einsum('ijk, j, k->i', psi_tmp, A_1[:, i], B_1) + phi_tmp @ C[:, i]
    
    for i in range(6, 9):
        psi[:, i, 9] = psi_tmp[:, :, i - 3] @ B_1 + phi_tmp @ D[:, i - 3]
        psi[:, 9, i] = psi_tmp[:, :, i - 3] @ B_1 + phi_tmp @ D[:, i - 3]

    psi[:, 9, 9] = np.einsum('ijk, j, k->i', psi_tmp, B_1, B_2) + phi_tmp @ B_2

    return phi, psi


def propagate_with_impulse(t_eval, X):
    r0 = X[:3]
    v0 = X[3:6]
    dv = X[6:9]
    t_impulse = X[9]
    t_eval = np.array(t_eval)

    # Separate times before and after impulse
    t_pre = t_eval[t_eval <= t_impulse]
    t_post = t_eval[t_eval > t_impulse]
    if len(t_pre) == 1:
        t_pre = np.hstack((t_pre, t_impulse))

    # Propagate before impulse
    if len(t_pre) > 0:
        sol_pre = solve_ivp(
            dynamics,
            [t_pre[0], t_pre[-1]],
            np.concatenate([r0, v0]),
            t_eval=t_pre,
            rtol=1e-8,
            atol=1e-10,
            method='DOP853'
        )
        state_at_impulse = sol_pre.y[:, -1]
    else:
        state_at_impulse = np.concatenate([r0, v0])
        sol_pre = None

    # Apply impulse (Δv)
    state_after_impulse = np.concatenate([state_at_impulse[:3], state_at_impulse[3:] + dv])

    if sol_pre:
        sol_pre.y[:, -1] = state_after_impulse

    # Propagate after impulse
    if len(t_post) > 0:
        sol_post = solve_ivp(
            dynamics,
            [t_post[0], t_post[-1]],
            state_after_impulse,
            t_eval=t_post,
            rtol=1e-8,
            atol=1e-10,
            method='DOP853'
        )
    else:
        sol_post = None

    # Stitch the two segments
    trajectory = []
    if sol_pre and len(sol_pre.y) > 0:
        trajectory.extend(sol_pre.y.T)
    if sol_post and len(sol_post.y) > 0:
        trajectory.extend(sol_post.y.T)

    return np.array(trajectory)

In [64]:
def generate_input_points(a, e, i, raan, argp, nu, noise, dvx, dvy, dvz, man_time):
    # Sample orbit test
    epoch = Time("2025-01-01 00:00:00", scale="utc")
    a = (a + R_e) * u.km
    e *= u.one
    i *= u.deg
    raan *= u.deg
    argp *= u.deg
    nu *= u.deg
    dv = [dvx/1000, dvy/1000, dvz/1000] * u.km / u.s

    orbit = Orbit.from_classical(Earth, a, e, i, raan, argp, nu, epoch)
    times = time_range(epoch, end=epoch + 1800 * u.s, num_values=181)

    positions = []
    velocities = []
    man_applied = False

    for t in times:
        dt = (t - epoch).sec
        orbit = orbit.propagate(10 * u.s)
        if dt >= man_time and not man_applied:
            # Apply impulse
            new_velocity = orbit.v + dv
            orbit = Orbit.from_vectors(Earth, orbit.r, new_velocity, epoch + dt * u.s)
            man_applied = True

        r = orbit.r.to_value(u.km)
        v = orbit.v.to_value(u.km / u.s)

        # Add Gaussian noise
        r_noisy = r + np.random.normal(0, noise, size=3)
        v_noisy = v + np.random.normal(0, noise, size=3)

        positions.append(r_noisy)
        velocities.append(v_noisy)

    positions = np.array(positions)
    velocities = np.array(velocities)
    states = np.hstack((positions, velocities))
    return states

observer = generate_input_points(500, 0.01, 45.05, 29.93, 132.9, -107.74, 0, 0, 0, 0, 0)
target = generate_input_points(1000, 0.02, 45, 94.80, 199.00, -54.13, 1e-5, 100, 100, 100, 920)

# Set initial estimation and covariance
X_0 = np.hstack((target[0], np.array([100/1000, 200/1000, 500/1000, 880])))
P_0 = np.diag([10**2, 10**2, 10**2, 1e-3**2, 1e-3**2, 1e-3**2, 5e-3**2, 5e-3**2, 5e-3**2, 50**2])**2

X_i = X_0
P_i = P_0

expected_points = propagate_with_impulse(t_eval, X_i)

z_tau = np.zeros((K, MEASUREMENT_DIM))
z_exp = np.zeros((K, MEASUREMENT_DIM))
for i in range(len(observer)):
    z_tau[i] = transform_state(target[i], observer[i])
    z_exp[i] = transform_state(expected_points[i], observer[i])

# Covariance matrix
sigma_noise = 1e-4
R = sigma_noise**2 * np.eye(K * MEASUREMENT_DIM)

In [5]:
# Propagate orbit and find modified STT
phi, psi = compute_stt(t_eval, X_i, propagate_with_impulse)

print(phi, psi)

[[-1.88253140e-01 -1.30274447e+00  4.22782739e-02  1.35350637e+03
  -1.10140581e+03 -3.08885391e+02  9.31776975e+02 -1.77303525e+02
  -1.27944653e+02 -6.51435239e+02]
 [ 2.13389823e-01  4.35037954e+00 -6.14370164e-01 -5.50886081e+02
   3.39980213e+03  2.93612152e+02 -1.61639526e+02  9.70260309e+02
   1.24513119e+02  9.54005484e+02]
 [-9.25592106e-02  7.67011199e-01 -2.45214037e-01 -3.47378735e+02
   7.74009421e+02  1.14985823e+03 -1.28260324e+02  1.36965614e+02
   8.65373670e+02  1.00310094e+03]
 [-1.02683432e-03 -3.73822550e-03  3.60220510e-04  1.11572003e+00
  -2.97146672e+00 -1.05987622e+00  1.24847168e+00 -5.90683889e-01
  -5.67587947e-01 -1.63701604e+00]
 [-8.28262308e-05  2.67148631e-03 -2.43762667e-04 -1.19341921e+00
   2.48592277e+00  8.19495792e-01 -4.84374414e-01  9.89419741e-01
   3.75863035e-01  1.15322758e+00]
 [ 3.56450957e-05  2.90721181e-03 -1.28092898e-03 -1.16811980e+00
   2.30017658e+00  4.38149046e-01 -5.69724979e-01  4.60356259e-01
   9.62507557e-01  1.02932989e+00

In [80]:
def calculate_U_and_Q(x_nom, observer_k, measurement_model, epsilon=np.array([10, 10, 10, 1, 1, 1])):
    d = MEASUREMENT_DIM
    U = np.zeros((d, 6))
    Q = np.zeros((d, 6, 6))

    for i in range(6):  # Perturb each state component
        dx = np.zeros(6)
        dx[i] = epsilon[i]

        z_plus = measurement_model(x_nom + dx, observer_k)
        z_minus = measurement_model(x_nom - dx, observer_k)

        U[:, i] = (z_plus - z_minus) / (2 * epsilon[i])

        for j in range(6):
            dxj = np.zeros(6)
            dxj[j] = epsilon[j]
            z_pp = measurement_model(x_nom + dx + dxj, observer_k)
            z_pm = measurement_model(x_nom + dx - dxj, observer_k)
            z_mp = measurement_model(x_nom - dx + dxj, observer_k)
            z_mm = measurement_model(x_nom - dx - dxj, observer_k)

            Q[:, i, j] = (z_pp - z_pm - z_mp + z_mm) / (4 * epsilon[j]**2)

    return U, Q

U = []
Q = []

for k in range(len(target)):
    x_k = expected_points[k]
    observer_k = observer[k]
    U_k, Q_k = calculate_U_and_Q(expected_points[0], observer[0], transform_state)
    U.append(U_k)
    Q.append(Q_k)

U = np.array(U)
Q = np.array(Q)


In [81]:
Xi = np.einsum('ijk,kp->ijp', U, phi)
Theta = np.einsum('ijk,kpq->ijpq', U, psi) + np.einsum('ijkl,kp,lq->ijpq', Q, phi, phi)
Theta = Theta.reshape(K * MEASUREMENT_DIM, PARAM_DIM, PARAM_DIM)
Xi = Xi.reshape(K * MEASUREMENT_DIM, PARAM_DIM)
m = 0.5 * np.einsum('iab,ab->i', Theta, P_i)

term1 = np.einsum('jpq,ab,pq->jab', Theta, P_i, P_i)
term2 = np.einsum('jpq,ap,bq->jab', Theta, P_i, P_i)
term3 = np.einsum('jpq,aq,bp->jab', Theta, P_i, P_i)
P_global = np.einsum('ia, jq, aq -> ij', Xi, Xi, P_i) - np.outer(m, m) + 0.25 * np.einsum('iab,jab->ij', (term1 + term2 + term3), Theta)

print(np.linalg.norm(Xi))
print(np.linalg.norm(Theta))

9.784909948538683
527381.1296479438


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Theta: (6, 10, 10)
Theta_row_norms = np.linalg.norm(Theta[-1, :, :].reshape(6, -1), axis=1)

plt.figure(figsize=(8, 5))
plt.bar(range(6), Theta_row_norms)
plt.xlabel("Measurement index (i)")
plt.ylabel("‖Theta[i,:,:]‖")
plt.title("Norm of Θ per Measurement Channel")
plt.grid(True)
plt.show()


ValueError: cannot reshape array of size 36200 into shape (6,newaxis)

In [11]:
delta_X_hat = tmp - 0.5 * np.linalg.pinv(Omega_linear) @ np.einsum('mij,i,j->m', Sigma_linear, tmp, tmp)

NameError: name 'tmp' is not defined

In [ ]:
Sigma = Theta.reshape(K * MEASUREMENT_DIM, PARAM_DIM, PARAM_DIM)
Omega = Xi.reshape(K * MEASUREMENT_DIM, PARAM_DIM)

W = np.linalg.inv(P_global + R)
W_norm = W / np.linalg.norm(W)
delta_z = (z_tau - z_exp).reshape(181 * MEASUREMENT_DIM)

H = Omega.T @ W_norm @ Omega
b = Omega.T @ W_norm @ delta_z
delta_X_linear = np.linalg.pinv(H) @ b

Gamma_bar = Omega + np.einsum('ikp,p->ik', Sigma, delta_X_linear)
delta_Z_linear = -2 * Gamma_bar.T @ W_norm @ delta_z
Omega_linear = -2 * Gamma_bar.T @ W_norm @ Omega

H_inv = np.linalg.pinv(H)
tmp = np.linalg.pinv(Omega_linear) @ delta_Z_linear
P_dx = np.linalg.pinv(Gamma_bar.T @ W_norm @ Omega) @ Gamma_bar.T @ W @ R @ W_norm.T @ Gamma_bar @ np.linalg.pinv(Omega.T @ W_norm @ Gamma_bar)

# Update estimate
#X_i += delta_X_linear
#P_i = P_dx
print(delta_X_hat)
print(X_i + delta_X_linear)
print(P_i)
print(P_dx)
print(np.linalg.norm(delta_X_linear))

#Convergence check
# if np.linalg.norm(delta_X_linear) <= nu:
#     print("Successfully converged!")
#     break

LinAlgError: Singular matrix

In [138]:
print(np.linalg.norm(Omega))
print(np.linalg.norm(Sigma))

for j in range(PARAM_DIM):
    print(f"‖phi[:, {j}]‖ = {np.linalg.norm(phi[:, j]):.3e}")

for j in range(PARAM_DIM):
    for k in range(PARAM_DIM):
        norm_psi_jk = np.linalg.norm(psi[:, j, k])
        if norm_psi_jk > 1e2:  # heuristic threshold
            print(f"High curvature: ‖psi[:, {j},{k}]‖ = {norm_psi_jk:.3e}")


8.010251216760603
372703.5790014552
‖phi[:, 0]‖ = 2.992e-01
‖phi[:, 1]‖ = 4.606e+00
‖phi[:, 2]‖ = 6.628e-01
‖phi[:, 3]‖ = 1.502e+03
‖phi[:, 4]‖ = 3.657e+03
‖phi[:, 5]‖ = 1.226e+03
‖phi[:, 6]‖ = 9.544e+02
‖phi[:, 7]‖ = 9.958e+02
‖phi[:, 8]‖ = 8.836e+02
‖phi[:, 9]‖ = 1.530e+03
High curvature: ‖psi[:, 0,9]‖ = 1.119e+04
High curvature: ‖psi[:, 1,9]‖ = 5.209e+04
High curvature: ‖psi[:, 2,9]‖ = 3.952e+04
High curvature: ‖psi[:, 3,9]‖ = 5.435e+07
High curvature: ‖psi[:, 4,3]‖ = 1.072e+02
High curvature: ‖psi[:, 4,4]‖ = 2.722e+02
High curvature: ‖psi[:, 4,5]‖ = 1.156e+02
High curvature: ‖psi[:, 4,9]‖ = 7.625e+07
High curvature: ‖psi[:, 5,9]‖ = 5.100e+07
High curvature: ‖psi[:, 6,1]‖ = 1.072e+02
High curvature: ‖psi[:, 6,9]‖ = 7.006e+04
High curvature: ‖psi[:, 7,1]‖ = 2.722e+02
High curvature: ‖psi[:, 7,9]‖ = 7.007e+04
High curvature: ‖psi[:, 8,1]‖ = 1.156e+02
High curvature: ‖psi[:, 8,9]‖ = 7.009e+04
High curvature: ‖psi[:, 9,0]‖ = 1.119e+04
High curvature: ‖psi[:, 9,1]‖ = 5.209e+04
High curva

In [127]:
print(f"‖phi[:, 9]‖ = {np.linalg.norm(phi[:, 9]):.3e}")

‖phi[:, 9]‖ = 1.301e+01
